In [526]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from requests import delete

In [527]:
input_file = "Student_Dataset.csv"
output_file = "Cleaned_Student_Dataset.csv"

df = pd.read_csv(input_file, delimiter=",", header=0, index_col=0, dtype=str)
first_len = len(df)
print("Len before claening :", first_len)

Len before claening : 1010


###### drop duplicate

In [528]:
before_count = len(df)
df = df.drop_duplicates()
after_count = len(df)
print(after_count)


df.info()

1001
<class 'pandas.DataFrame'>
Index: 1001 entries, 2 to 1011
Data columns (total 2 columns):
 #   Column                                                                                                                         Non-Null Count  Dtype
---  ------                                                                                                                         --------------  -----
 0   1                                                                                                                              1001 non-null   str  
 1   On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step.  606 non-null    str  
dtypes: str(2)
memory usage: 23.5+ KB


#### get sentences

In [529]:
sentences = []

for index, row in df.iterrows():
    sentence = None

    col1 = row.iloc[1] if len(row) > 1 else None
    col2 = row.iloc[2] if len(row) > 2 else None

    if pd.notna(col1) and str(col1).strip() == "0x000000":
        # Sentence is in col2
        if pd.notna(col2):
            sentence = str(col2).strip()
    elif pd.notna(col1):
        # Sentence is in col1
        sentence = str(col1).strip()
    elif pd.notna(col2):
        # Fallback: sentence only in col2
        sentence = str(col2).strip()
    if sentence:
        sentences.append(sentence)

print("len total :", len(sentences))
print ("lost :", after_count - len(sentences), "Lines")


len total : 606
lost : 395 Lines


#### delete useless words
example in the sentence : "I have a persistent feeling of thirst, especially after meals. I also lose weight without trying."
we keep : "persistent feeling thirst, especially after meals.  lose weight without trying."

In [530]:
stop_words = {
    'the', 'a', 'an', 'and', 'or', 'but', 'if', 'then', 'else', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'to', 'of', 'in', 'for', 'with',
    'on', 'at', 'by', 'from', 'up', 'down', 'this', 'that', 'it', 'its', 'i'
}

cleaned_sentences = []

for s in sentences:
    raw_sentence = s
    words = raw_sentence.lower().split()
    filtered_words = [w for w in words if w not in stop_words]
    clean_sentence = " ".join(filtered_words)
    cleaned_sentences.append(clean_sentence)


Create new dataframe without duplicate rows or missing value

In [531]:
df_final = pd.DataFrame({
    "Color": ["0x000000"] * len(cleaned_sentences),
    "Sentences": cleaned_sentences
})

print("Before :", first_len)
print("After :", len(df_final))
print("Lost : ", first_len - len(df_final), "Lines")

Before : 1010
After : 606
Lost :  404 Lines


#### Put the dataframe in a csv file

In [532]:
df_final.to_csv(output_file, index=True, index_label="Index")